In [16]:
import logging 
import pandas as pd
from haystack.utils import fetch_archive_from_http
from haystack import Document
from haystack.document_stores import FAISSDocumentStore
from haystack.nodes import RAGenerator, DensePassageRetriever
from haystack.pipelines import GenerativeQAPipeline
from haystack.utils import print_answers

logging.basicConfig(format="%(levelname)s - %(name)s - %(message)s, level=logging.WARNING")
logging.getLogger("haystack").setLevel(logging.INFO)

In [9]:
doc_dir="."
s3_url = "https://s3.eu-central-1.amazonaws.com/deepset.ai-farm-qa/datasets/small_generator_dataset.csv.zip"
fetch_archive_from_http(url=s3_url, output_dir=doc_dir)

# Criar dataframe com colunas "titulo" e "texto"
df = pd.read_csv(f"{doc_dir}/small_generator_dataset.csv", sep=",")
# Limpeza minima
df.fillna(value="", inplace=True)

print(df.head())
                 

INFO - haystack.utils.import_utils - Found data stored in '.'. Delete this first if you really want to fetch new data., level=logging.WARNING


               title                                               text
0  "Albert Einstein"  to Einstein in 1922. Footnotes Citations Alber...
1  "Albert Einstein"  Albert Einstein Albert Einstein (; ; 14 March ...
2  "Albert Einstein"  observations were published in the internation...
3  "Albert Einstein"  model for depictions of mad scientists and abs...
4     "Alfred Nobel"  was adopted as the standard technology for min...


In [11]:
## Podemos atribuir os dados a um objeto de Haystack Document, ou
##      podemos apenas usar dicionários com campos "texto" e "meta"

titles = list(df["title"].values)
texts = list(df["text"].values)
documents=[]

for title, text in zip(titles,texts):
    documents.append(Document(content=text, meta={"name": title or ""}))

In [22]:
## Iniciar o Gerador
generator = RAGenerator(
    model_name_or_path="facebook/rag-token-nq",
    use_gpu=True,
    top_k=1,
    max_length=200,
    min_length=2,
    embed_title=True, # Serve para que o gerador saiba que o título é uma parte do texto
    num_beams=2 # Beams para a busca; beams são hipóteses que o modelo considera
)

""" NOTA: VER PromptNode -> substitui o RAGenerator já que está em desuso
        e nas versões mais recentes já nem existe"""


INFO - haystack.modeling.utils - Using devices: CPU - Number of GPUs: 0, level=logging.WARNING
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer c

pytorch_model.bin:   0%|          | 0.00/2.06G [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/rag-token-nq were not used when initializing RagTokenForGeneration: ['rag.question_encoder.question_encoder.bert_model.pooler.dense.weight', 'rag.question_encoder.question_encoder.bert_model.pooler.dense.bias']
- This IS expected if you are initializing RagTokenForGeneration from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RagTokenForGeneration from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [23]:
## Escrita dos documentos
##  escrevemos para a documentStore, primeiro apagando qualquer documento que 
##  ja la exista e depois criamos um embedder para cada documento

document_store.delete_documents()
document_store.write_documents(documents)
document_store.update_embeddings(retriever=retriever)

## Iniciar o pipeline
#   com uma pipeline podemos juntar os blocos numa pipeline de pesquisa
##  as pipelines são DAGs (Directed Acylic Graphs) que facilmente personalizam-se
##      para os casos de uso que queremos, sendo que o haystack já vem com umas

pipe = GenerativeQAPipeline(generator=generator, retriever=retriever)



Writing Documents:   0%|          | 0/75 [00:00<?, ?it/s]

INFO - haystack.document_stores.faiss - Updating embeddings for 68 docs..., level=logging.WARNING


Updating Embedding:   0%|          | 0/68 [00:00<?, ? docs/s]

Create embeddings:   0%|          | 0/80 [00:00<?, ? Docs/s]

In [38]:
## Questões

QUESTIONS = [
    "What's the name of the senior vice president at apple in 2014?",
]

for question in QUESTIONS:
    res = pipe.run(query=question, params={"Generator":{"top_k":1}, 
                                           "Retriever":{"top_k":5}})
    print_answers(res, details="minimum")

C:\Users\h_e_m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\transformers\generation\utils.py:2831: UserWarning: `max_length` is deprecated in this function, use `stopping_criteria=StoppingCriteriaList(MaxLengthCriteria(max_length=max_length))` instead.
  warnings.warn(


"Query: What's the name of the senior vice president at apple in 2014?"
'Answers:'
[{'answer': ' angela ahrendts'}]
